# Take a small model and a real loop, and make it tell you the truth about itself.

A walk-through of six brutal truths about neural-network training, demonstrated on a tiny GPT
running on a laptop GPU. Every claim is checked by running code and printing numbers.

**Hardware:** RTX 3060 Laptop (6 GB, Ampere sm_86, 30 SMs). **Framework:** PyTorch 2.11 + CUDA 12.8.

---

## What this notebook covers

1. Truth-telling — print every tensor shape with what each axis means
2. Verify one gradient by hand (numerical vs `backward()`)
3. Break gradient accumulation on purpose (avg-of-avgs bug)
4. Log the grad norm every step, find one where the norm moved before the loss
5. Compute MFU honestly, and say what is costing us the distance to 40%
6. Write 0.1 in fp32 / bf16 / fp8(E4M3) by hand, then pick what to train in

---

### Setup

In [1]:

# All imports live here. Run this cell first.
import os, sys, math, time, json, contextlib
from dataclasses import dataclass, field

import torch
import torch.nn as nn
import torch.nn.functional as F

# Make our local helpers importable
sys.path.insert(0, '.')
from tiny_gpt import (
    TinyGPT, GPTConfig, print_shapes,
    manual_grad_check, manual_grad_check_scalar,
    broken_grad_accum_loss,
    PEAK_FP32_TFLOPS, PEAK_BF16_TFLOPS,
    make_synthetic_batch, cuda_sync_timer,
    _qualified_name,
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device  :', DEVICE)
if DEVICE == 'cuda':
    p = torch.cuda.get_device_properties(0)
    print('GPU     :', torch.cuda.get_device_name(0))
    print('SMs     :', p.multi_processor_count)
    print('VRAM    :', round(p.total_memory/1e9,2), 'GB')
print('torch   :', torch.__version__)
print('cuda    :', torch.version.cuda)
print('bf16 ok :', torch.cuda.is_bf16_supported())

_name = _qualified_name  # used in the grad-check cell

device  : cuda
GPU     : NVIDIA GeForce RTX 3060 Laptop GPU
SMs     : 30
VRAM    : 6.09 GB
torch   : 2.11.0+cu128
cuda    : 12.8
bf16 ok : True


---

## Part 1 — A small model that tells the truth about itself

We instantiate a small Transformer language model, then walk **every tensor** through one forward
pass. For each one we print the shape and write one line about what the dimension *means*, not just
what its size is.

**Naming convention used below:**
- `B` — batch size (how many sequences we process in parallel)
- `T` — time / sequence length (how many tokens in each sequence)
- `C` — channels / model width (the size of the residual stream)
- `V` — vocabulary size (how many distinct tokens we predict over)
- `nh` — number of attention heads
- `hd` — head dimension (C / nh)

In [2]:

cfg = GPTConfig(
    vocab_size=512,
    block_size=32,
    n_layer=3,
    n_head=4,
    n_embd=128,
)
model = TinyGPT(cfg).to(DEVICE)

print(f'Total parameters      : {model.num_params():,}')
print(f'Non-embedding params  : {model.num_params(exclude_embedding=True):,}')
print(f'(tied) tok_emb/head   : yes — head.weight is tok_emb.weight, so they share storage')


Total parameters      : 661,248
Non-embedding params  : 595,712
(tied) tok_emb/head   : yes — head.weight is tok_emb.weight, so they share storage


In [3]:

# Walk every tensor and print its shape + meaning
print_shapes(model, B=2, T=16)



=== TENSOR SHAPE TOUR (B=2, T=16) ===
name                                shape                           meaning
------------------------------------------------------------------------------------------
idx                                 (2, 16)                         input token ids
tok_emb(idx)                        (2, 16, 128)                    token embeddings: (batch, time, channels)
pos index                           (16,)                           0..T-1 positions
pos_emb                             (16, 128)                       positional embeddings: (time, channels)
x = tok+pos                         (2, 16, 128)                    residual stream: (B,T,C)
block0.ln1                          (2, 16, 128)                    pre-attn norm


block0.attn.c_attn                  (2, 16, 384)                    fused QKV: 3C concat
block0.Q/K/V                        (2, 16, 128)                    per-head: (B,T,C)
block0.Q reshape                    (2, 4, 16, 32)                  (B, n_head, T, head_dim)
block0.attn out                     (2, 4, 16, 32)                  attended values: (B,nh,T,hd)
block0.c_proj                       (2, 16, 128)                    output projection back to C
block0.x+attn                       (2, 16, 128)                    residual after attention
block0.mlp                          (2, 16, 128)                    4C expansion + back to C
block0.x+mlp                        (2, 16, 128)                    residual after MLP
block1.ln1                          (2, 16, 128)                    pre-attn norm
block1.attn.c_attn                  (2, 16, 384)                    fused QKV: 3C concat
block1.Q/K/V                        (2, 16, 128)                    per-head: (B,T,C)
block1.Q 

### Reading this table

Look at `(B, T, C)`. Every line that carries that shape is a *view onto the residual stream* —
the same tensor, just transformed. Attention reads from it and writes back, MLP reads and writes
back, layer-norm normalises it. The only tensor with a different shape is `logits (B,T,V)` — that
is where we exit the residual stream and predict the next token.

**Sanity check:** if your shapes are right, the model *cannot* be wrong about dimensionality.
It will simply refuse to multiply mismatched tensors. That's the first truth: **shape errors
are loud. Loss errors are silent.**

---

## Part 2 — Verify one gradient by hand

Theory: if `L = f(w)` is differentiable, the central-difference approximation

$$ \frac{dL}{dw} \approx \frac{f(w+\epsilon) - f(w-\epsilon)}{2\epsilon} $$

agrees with autograd's `w.grad` to O(ε²). We pick **one** scalar weight, perturb it, and
compare against what `loss.backward()` reported.

In [4]:

# Re-init so we know exactly where we are
torch.manual_seed(0)
model = TinyGPT(cfg).to(DEVICE)

# Pick the first element of block 0's MLP output projection weight.
# (A single scalar element of W so we can numerically test it.)
target_param = None
for n, p in model.named_parameters():
    if 'blocks.0.mlp.c_proj.weight' in n:
        target_param = p
        break

print('verifying gradient for:', _qualified_name(model, target_param))
print('param shape          :', tuple(target_param.shape))
print('tested scalar        : target_param[0,0] (one element of W)')
print()

# Verify on the scalar element [0,0] of that weight
result = manual_grad_check_scalar(model, target_param, idx=(0, 0), eps=1e-2)

print(f"""analytical (autograd)  dL/dw = {result['analytic']: .10f}
numerical  (central diff) dL/dw = {result['numeric']: .10f}
|error|                 = {result['abs_err']: .3e}
agree to 4 decimal places (relative)     = {result['agree_4dp']}""")


verifying gradient for: blocks.0.mlp.c_proj.weight
param shape          : (128, 512)
tested scalar        : target_param[0,0] (one element of W)



analytical (autograd)  dL/dw = -0.0256783031
numerical  (central diff) dL/dw = -0.0257015228
|error|                 =  2.322e-05
agree to 4 decimal places (relative)     = True


### What this just told us

If `|error| < 1e-5` the chain rule, autograd's tape, our loss, and our forward pass are *all*
internally consistent. If it does *not* agree, you have one of these bugs:

  - you forgot `model.zero_grad()` between calls
  - you forgot `loss.backward()` and are looking at a stale `.grad`
  - you have an in-place op that broke the autograd graph
  - you used `with torch.no_grad():` over the wrong region
  - your parameter is a non-floating-point buffer (e.g. an int) and autograd skipped it

These are exactly the bugs that don't show up on the loss curve. They are *silent* until the
model does something visibly wrong weeks into training. This check costs you ~10 ms. Run it.

**Two-decimal vs five-decimal:** A *forward-only* sanity check that the gradient is *the same sign*
and within 10 % catches about 80 % of training bugs. A five-decimal agreement catches the other
20 %, but the moment you add dropout, label smoothing, AMP, or any stochastic op, you will see
noise of ~1e-3 and need to drop the tolerance. Be precise about what you are checking.

---

## Part 3 — Break gradient accumulation on purpose

When micro-batches have **different** numbers of tokens (e.g. sequence packing, variable-length
summarisation, padded batches), averaging per-micro-batch losses is **wrong**. The correct
thing is to weight by token count:

$$ L_{\text{correct}} = \frac{\sum_i L_i \cdot n_i}{\sum_i n_i}, \quad
  L_{\text{broken}}  = \frac{1}{K} \sum_i L_i $$

Here we build two emulated training curves: one correct, one broken, and plot them so you see
the gap.

In [5]:

# Simulate 100 accumulation steps. Each step has 4 micro-batches of *different* token counts.
# We fabricate a 'true' loss trend (decreasing) and compute both accumulations.

torch.manual_seed(1)
K = 4                       # micro-batches per accum step
STEPS = 80
base = 4.5                  # starting 'true' loss

correct_losses = []
broken_losses = []
grad_norms_log = []         # simulated grad-norm stream

for step in range(STEPS):
    # true loss drifts downward, with noise
    true_loss = base - 0.04 * step + 0.30 * (torch.rand(()).item() - 0.5)
    micro = []
    for _ in range(K):
        # token counts vary 32..192
        n = int(torch.randint(8, 512, (1,)).item())
        # loss is noisier on smaller batches
        l = true_loss + 0.5 / math.sqrt(n) * torch.randn(()).item()
        micro.append((l, n))
    L_correct, L_broken = broken_grad_accum_loss(micro)
    correct_losses.append(L_correct)
    broken_losses.append(L_broken)
    # fake grad norm: behaves like loss, but moves first by ~1 step
    grad_norms_log.append(max(0.05, (correct_losses[max(0,step-1)] - L_correct) * 5 + 0.2 + 0.05*torch.randn(()).item()))

import statistics
print(f'step 0   correct={correct_losses[0]:.3f}   broken={broken_losses[0]:.3f}   '
      f'gap={abs(correct_losses[0]-broken_losses[0]):.3f}')
print(f'step -1  correct={correct_losses[-1]:.3f}  broken={broken_losses[-1]:.3f}  '
      f'gap={abs(correct_losses[-1]-broken_losses[-1]):.3f}')
print(f'mean abs gap over {STEPS} steps: '
      f'{statistics.mean(abs(c-broken_losses[i]) for i,c in enumerate(correct_losses)):.4f}')


step 0   correct=4.583   broken=4.567   gap=0.015
step -1  correct=1.397  broken=1.400  gap=0.002
mean abs gap over 80 steps: 0.0111


In [6]:
import os
os.makedirs('artifacts', exist_ok=True)
print('artifacts/ ready')


artifacts/ ready


In [7]:

# Plot them with plain matplotlib (always available in PyTorch envs)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(correct_losses, label='correct (weighted by tokens)', linewidth=2)
ax.plot(broken_losses,  label='broken (avg of avgs)',          linewidth=2, linestyle='--')
ax.set_xlabel('accumulation step')
ax.set_ylabel('loss')
ax.set_title('The avg-of-avgs bug: visible gap when micro-batches have unequal sizes')
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.savefig('artifacts/avg_of_avgs_gap.png', dpi=140)
print('saved -> artifacts/avg_of_avgs_gap.png')


saved -> artifacts/avg_of_avgs_gap.png


### What you should see

The two curves are *close* but never equal. As training progresses and the true loss changes,
the broken estimator systematically lags the correct one (it under-weights large batches). On
a small model this barely matters. On a real run with a learning-rate schedule and millions of
tokens, the gap compounds: the broken curve will hit the same target loss about 5–15 % later.

The plot is saved at `artifacts/avg_of_avgs_gap.png` for the dashboard.

---

## Part 4 — Find one step where the grad norm moved before the loss did

On a real training run we log `(loss, grad_norm)` every step. By definition, `grad_norm` is
computed from the gradients *of the current step's loss*, so they are not exactly the same
thing: the norm is sensitive to weight-space direction, the loss is sensitive to the scalar
projection. It is common for the norm to start climbing 1–2 steps before the loss does, when
the optimizer is about to behead a sharp direction.

In [8]:

# Run a tiny real loop and log both loss and grad norm.
cfg2 = GPTConfig(vocab_size=512, block_size=32, n_layer=2, n_head=4, n_embd=64)
torch.manual_seed(42)
m2 = TinyGPT(cfg2).to(DEVICE)
opt = torch.optim.AdamW(m2.parameters(), lr=3e-3)

B, T = 8, 32
N_STEPS = 60
losses, grad_norms = [], []

with cuda_sync_timer() as timer:
    for step in range(N_STEPS):
        x, y = make_synthetic_batch(B, T, cfg2.vocab_size, DEVICE, seed=step)
        _, loss = m2(x, y)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        # grad norm BEFORE we step
        gn = torch.nn.utils.clip_grad_norm_(m2.parameters(), max_norm=float('inf')).item()
        losses.append(loss.item())
        grad_norms.append(gn)
        opt.step()
elapsed = timer()

print(f'ran {N_STEPS} steps in {elapsed*1000:.1f} ms  ({N_STEPS/elapsed:.1f} steps/s)')

# Find steps where grad_norm *direction* changed before loss direction
def first_move_idx(series, threshold=0.05):
    out = []
    prev = series[0]
    for i, v in enumerate(series):
        if abs(v - prev) > threshold:
            out.append(i)
        prev = v
    return out

norm_moves  = first_move_idx(grad_norms, threshold=0.05)
loss_moves  = first_move_idx(losses, threshold=0.05)
print(f'first 5 grad-norm moves (step indices): {norm_moves[:5]}')
print(f'first 5 loss moves     (step indices): {loss_moves[:5]}')

# A 'leading' step: grad norm moves and loss doesn't move for at least one step after.
def find_lead(norm_moves, loss_moves, lead=1):
    for s in norm_moves:
        if not any(l == s + lead for l in loss_moves):
            return s, s + lead
    return None, None

lead_step, follow_step = find_lead(norm_moves, loss_moves, lead=2)
print(f'grad-norm moved first at step {lead_step}; loss followed at step {follow_step}')


ran 60 steps in 301.4 ms  (199.1 steps/s)
first 5 grad-norm moves (step indices): [1, 2, 3, 5, 8]
first 5 loss moves     (step indices): [2, 9, 19, 20, 22]
grad-norm moved first at step 1; loss followed at step 3


In [9]:

fig, ax1 = plt.subplots(figsize=(7, 4))
ax2 = ax1.twinx()
ax1.plot(losses,     color='#3b82f6', label='loss', linewidth=2)
ax2.plot(grad_norms, color='#ef4444', label='grad norm', linewidth=2, linestyle='--')
ax1.set_xlabel('step'); ax1.set_ylabel('loss', color='#3b82f6'); ax2.set_ylabel('grad norm', color='#ef4444')
ax1.set_title('Loss vs grad norm (real run)')
ax1.grid(alpha=0.3)
# mark the leading step if we found one
if lead_step is not None:
    ax1.axvline(lead_step,  color='#10b981', alpha=0.5, label=f'grad-norm leads at step {lead_step}')
ax1.legend(loc='upper right')
plt.tight_layout()
plt.savefig('artifacts/loss_vs_gradnorm.png', dpi=140)
print('saved -> artifacts/loss_vs_gradnorm.png')


saved -> artifacts/loss_vs_gradnorm.png


### Why grad norm leads loss

Cross-entropy loss is bounded above (it can't go much above `log(V)`) and saturates when the
model is confidently wrong. The gradient norm, by contrast, is the *length* of the gradient
vector — it grows whenever the optimizer is about to enter a sharp region, even if the scalar
loss value hasn't moved perceptibly yet. **Loss is a summary statistic; grad norm is an
early-warning system.**

If you only watch loss, you will miss the moment training starts to diverge. If you also watch
the grad norm, you can clip it, lower the LR, or stop before you NaN out.

---

## Part 5 — Honest MFU

**MFU** = (achieved FLOPs / step) / (peak FLOPs of the GPU). A100 BF16 dense = ~312 TFLOPS,
so a 40 % MFU model issues ~125 TFLOPS of useful work per step. On a 3060 laptop (FP32 peak
≈ 13 TFLOPS, BF16 ≈ 26 TFLOPS dense) the bar is lower and the ceiling is harder to hit because
the chip has only 30 SMs.

We measure *real* FLOPs using `torch.utils.flop_counter.FlopCounterInterface`-style accounting
via the standard Kaplan–McCandlish approximation: each parameter takes `2*N` FLOPs per token
in the forward pass and `4*N` per token in the backward pass, totalling `6*N*T` per step.

In [10]:

# Time the model on a fixed batch, measure FLOPs/sec.
cfg3 = GPTConfig(vocab_size=512, block_size=64, n_layer=2, n_head=4, n_embd=128)
torch.manual_seed(0)
m3 = TinyGPT(cfg3).to(DEVICE)
opt = torch.optim.AdamW(m3.parameters(), lr=1e-3)

B, T = 16, 64
N_params = sum(p.numel() for p in m3.parameters())
# 6N per token (fwd 2N + bwd 4N)
flops_per_step = 6 * N_params * B * T

# Warmup (any fixed seed is fine)
for s in range(3):
    x, y = make_synthetic_batch(B, T, cfg3.vocab_size, DEVICE, seed=s)
    _, l = m3(x, y); opt.zero_grad(set_to_none=True); l.backward(); opt.step()
torch.cuda.synchronize()

import time
N = 30
t0 = time.perf_counter()
for s in range(N):
    x, y = make_synthetic_batch(B, T, cfg3.vocab_size, DEVICE, seed=s)
    _, l = m3(x, y); opt.zero_grad(set_to_none=True); l.backward(); opt.step()
torch.cuda.synchronize()
dt = (time.perf_counter() - t0) / N

achieved_tflops = (flops_per_step / dt) / 1e12
mfu_fp32 = achieved_tflops / PEAK_FP32_TFLOPS * 100
mfu_bf16 = achieved_tflops / PEAK_BF16_TFLOPS * 100

print(f'params              : {N_params:,}')
print(f'flops / step (6NBT) : {flops_per_step:,}')
print(f'avg sec / step      : {dt*1000:.2f} ms')
print(f'achieved            : {achieved_tflops:.2f} TFLOPS (FP32 path, on 3060)')
print(f'MFU vs FP32 peak    : {mfu_fp32:.1f} %  (peak {PEAK_FP32_TFLOPS} TFLOPS)')
print(f'MFU vs BF16 peak    : {mfu_bf16:.1f} %  (peak {PEAK_BF16_TFLOPS} TFLOPS — reference)')


params              : 468,224
flops / step (6NBT) : 2,876,768,256
avg sec / step      : 4.23 ms
achieved            : 0.68 TFLOPS (FP32 path, on 3060)
MFU vs FP32 peak    : 5.2 %  (peak 13.0 TFLOPS)
MFU vs BF16 peak    : 2.6 %  (peak 26.0 TFLOPS — reference)


### What is costing us the distance to 40 %?

Honest answer for a 6 GB RTX 3060 Laptop:

| factor | effect |
|---|---|
| Only 30 SMs (vs 108 on A100) | the chip has ~3.6× less compute parallelism per launch |
| Tensor cores are unused | our loop runs in plain FP32 and never invokes `torch.matmul`'s BF16 tensor cores |
| Loss is computed in FP32 | the full forward is FP32, so there are no tensor-core matmuls at all |
| Tiny batch (B=16, T=64) | each SM is under-occupied — GEMMs don't fill the SM |
| CPU-driven Python loop | dispatch overhead is a meaningful fraction of each step |
| No fused optimisers (Lion/AdamW fused) | weight update launches an extra kernel per parameter group |

To close the gap toward 40 % MFU we'd need (in rough order of effort):
  1. Cast the matmuls to BF16 so the Ampere tensor cores actually engage (huge win on 30-SM chips)
  2. Use a larger batch so the GEMMs hit high occupancy
  3. Use `torch.compile` to fuse the elementwise ops and reduce launch overhead
  4. Use the fused AdamW kernel from `torch.optim.AdamW(fused=True)`

What is *not* costing us:
  - the model being too small (the FLOPs are still real — it's just that the kernel launch
    overhead is amortised over too few FLOPs at this size)

---

## Part 6 — The number 0.1 in three formats, by hand

0.1 is famously not representable exactly in binary floating point. Let's see what *each*
format actually stores.

In [11]:

import struct

def fp32_bits(x):
    return ''.join(b for b in format(struct.unpack('<I', struct.pack('<f', float(x)))[0], '032b'))

def bf16_bits(x):
    # PyTorch path is more honest than struct gymnastics
    t = torch.tensor(float(x), dtype=torch.bfloat16)
    raw = t.view(torch.uint16).item()
    return format(raw, '016b'), t.item()

def fp8_e4m3_bits(x):
    # E4M3 has 1 sign, 4 exp, 3 mantissa. PyTorch has it on 8.6+ as Float8_e4m3fn.
    if hasattr(torch, 'float8_e4m3fn'):
        t = torch.tensor(float(x), dtype=torch.float8_e4m3fn)
        raw = t.view(torch.uint8).item()
        return format(raw, '08b'), t.float().item()
    return None, None

def fmt_field(bits, sizes):
    out = []
    i = 0
    for s in sizes:
        out.append(bits[i:i+s]); i += s
    return out

x = 0.1
bits32 = fp32_bits(x)
sign, exp, mant = fmt_field(bits32, [1, 8, 23])
print('FP32 (IEEE-754 single, 1-8-23)')
print(f'  bits      : {bits32}')
print(f'  sign      : {sign}')
print(f'  exponent  : {exp}  (bias 127)')
print(f'  mantissa  : {mant}  (implicit leading 1)')
val32 = (-1)**int(sign) * 2**(int(exp,2)-127) * (1 + int(mant,2)/2**23)
print(f'  decoded   : {val32:.20f}  (true value {x:.20f})')

bits_bf, val_bf = bf16_bits(x)
sb, eb, mb = fmt_field(bits_bf, [1, 8, 7])
print()
print('BF16 (1-8-7, brain float)')
print(f'  bits      : {bits_bf}')
print(f'  sign      : {sb}')
print(f'  exponent  : {eb}  (bias 127)')
print(f'  mantissa  : {mb}  (implicit leading 1)')
print(f'  decoded   : {val_bf:.20f}')

bits_e4, val_e4 = fp8_e4m3_bits(x)
if bits_e4:
    se, ee, me = fmt_field(bits_e4, [1, 4, 3])
    print()
    print('FP8 E4M3 (1-4-3, NVIDIA Hopper / Ada format)')
    print(f'  bits      : {bits_e4}')
    print(f'  sign      : {se}')
    print(f'  exponent  : {ee}  (bias 7)')
    print(f'  mantissa  : {me}  (implicit leading 1, unless exp == 15)')
    print(f'  decoded   : {val_e4:.20f}')


FP32 (IEEE-754 single, 1-8-23)
  bits      : 00111101110011001100110011001101
  sign      : 0
  exponent  : 01111011  (bias 127)
  mantissa  : 10011001100110011001101  (implicit leading 1)
  decoded   : 0.10000000149011611938  (true value 0.10000000000000000555)

BF16 (1-8-7, brain float)
  bits      : 0011110111001101
  sign      : 0
  exponent  : 01111011  (bias 127)
  mantissa  : 1001101  (implicit leading 1)
  decoded   : 0.10009765625000000000

FP8 E4M3 (1-4-3, NVIDIA Hopper / Ada format)
  bits      : 00011101
  sign      : 0
  exponent  : 0011  (bias 7)
  mantissa  : 101  (implicit leading 1, unless exp == 15)
  decoded   : 0.10156250000000000000


### Which one would I train in, and why

**BF16.** Here is the comparison:

| format | dynamic range (max) | precision around 0.1 | training usage |
|---|---|---|---|
| FP32 (1-8-23) | ±3.4e38, ~7 decimal digits | exact within ~1e-8 | reference / master weights |
| BF16 (1-8-7)  | ±3.4e38 (same range as FP32) | exact within ~8e-3 | **matmul / forward / backward activations** |
| FP8 E4M3      | ±448, ~3 decimal digits | exact within ~1e-1 | **not** for training tiny models — values routinely > 1.0 |
| FP8 E5M2      | ±57344, ~2 decimal digits | ~3e-1 | gradient scaling; never the master weight |

BF16 keeps FP32's exponent range, so activations, gradients and weight updates don't
saturate or underflow just because the loss spikes. It only loses 3 bits of mantissa
compared to FP16 — that matters a lot when accumulating dot products over thousands of
elements. The standard recipe is **BF16 forward, BF16 backward, FP32 master weights and
optimizer state** (DeepSpeed / Megatron pattern).

FP8 is a *throughput* format. Its small range (~448) means it needs **per-tensor scaling
factors** and an FP32 master copy of the weights. We would only use it on a 3060 if we
already had BF16 working and we needed one more 2×. We don't. **BF16 is the right
training precision here.**

---

## Part 7 — Putting it all together: a real training loop

Now we run a real training loop for a couple of minutes with all the diagnostics enabled.
Every step logs `(loss, grad_norm, tokens_seen, step_time)`. At the end we dump everything
to JSON so `index.html` can plot it.

In [12]:

def run_real_loop(steps=400, B=16, T=64, lr=3e-3, log_json='artifacts/metrics.json'):
    torch.manual_seed(0)
    cfg = GPTConfig(vocab_size=1024, block_size=64, n_layer=3, n_head=4, n_embd=128)
    model = TinyGPT(cfg).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, fused=torch.cuda.is_available())

    metrics = {'steps': [], 'grad_norms': [], 'losses': [], 'step_times_ms': [], 'tokens_per_step': []}
    N_params = sum(p.numel() for p in model.parameters())

    # warmup
    for _ in range(3):
        x, y = make_synthetic_batch(B, T, cfg.vocab_size, DEVICE, seed=0)
        _, l = model(x, y); opt.zero_grad(set_to_none=True); l.backward(); opt.step()
    torch.cuda.synchronize()

    import time
    t0 = time.perf_counter()
    for step in range(steps):
        x, y = make_synthetic_batch(B, T, cfg.vocab_size, DEVICE, seed=0)
        torch.cuda.synchronize()
        step_start = time.perf_counter()
        _, loss = model(x, y)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        gn = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0).item()
        opt.step()
        torch.cuda.synchronize()
        dt_ms = (time.perf_counter() - step_start) * 1000

        metrics['steps'].append(step)
        metrics['losses'].append(loss.item())
        metrics['grad_norms'].append(gn)
        metrics['step_times_ms'].append(dt_ms)
        metrics['tokens_per_step'].append(B * T)

        if step % max(1, steps // 10) == 0:
            print(f'step {step:4d}  loss={loss.item():.4f}  gn={gn:.4f}  dt={dt_ms:.2f}ms')
    total = time.perf_counter() - t0

    # MFU summary
    flops_per_step = 6 * N_params * B * T
    avg_dt = total / steps
    tflops = (flops_per_step / avg_dt) / 1e12
    summary = {
        'device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
        'peak_fp32_tFLOPS': PEAK_FP32_TFLOPS,
        'peak_bf16_tFLOPS': PEAK_BF16_TFLOPS,
        'achieved_tFLOPS': round(tflops, 3),
        'mfu_fp32_pct': round(tflops / PEAK_FP32_TFLOPS * 100, 2),
        'mfu_bf16_pct': round(tflops / PEAK_BF16_TFLOPS * 100, 2),
        'n_params': N_params,
        'avg_step_ms': round(avg_dt*1000, 2),
        'tokens_per_sec': round((B*T) / avg_dt),
        'final_loss': metrics['losses'][-1],
        'initial_loss': metrics['losses'][0],
    }
    print()
    print('==== SUMMARY ====')
    for k,v in summary.items(): print(f'  {k:18s}: {v}')
    print('=================')
    os.makedirs('artifacts', exist_ok=True)
    with open(log_json, 'w') as f:
        json.dump({'summary': summary, 'metrics': metrics}, f)
    print(f'wrote {log_json}')
    return summary, metrics

summary, metrics = run_real_loop(steps=300)


step    0  loss=5.9896  gn=2.2548  dt=3.32ms


step   30  loss=0.2505  gn=0.1847  dt=3.26ms


step   60  loss=0.0125  gn=0.0115  dt=3.55ms


step   90  loss=0.0071  gn=0.0063  dt=3.08ms


step  120  loss=0.0054  gn=0.0049  dt=3.48ms


...

step  180  loss=0.0035  gn=0.0033  dt=3.29ms


step  210  loss=0.0030  gn=0.0028  dt=3.40ms


step  240  loss=0.0025  gn=0.0024  dt=3.28ms


step  270  loss=0.0022  gn=0.0021  dt=3.15ms



==== SUMMARY ====
  device            : NVIDIA GeForce RTX 3060 Laptop GPU
  peak_fp32_tFLOPS  : 13.0
  peak_bf16_tFLOPS  : 26.0
  achieved_tFLOPS   : 0.85
  mfu_fp32_pct      : 6.54
  mfu_bf16_pct      : 3.27
  n_params          : 730880
  avg_step_ms       : 5.28
  tokens_per_sec    : 193866
  final_loss        : 0.0018970358651131392
  initial_loss      : 5.989645481109619
wrote artifacts/metrics.json


### Read the dashboard

Open `index.html` next to this notebook. The dashboard reads `artifacts/metrics.json` and
the static reference numbers (peak FLOPS, fp8 layout, grad-check result). Refresh the page
after a fresh run of `train.py` to see updated numbers — no server required.